# Put credit spread backtest for a $3,000 account

Tests the rule set for the tastytrade learning account: **$1-wide put credit spreads on liquid ETFs, 20-25 delta, 30-45 days out, sold when implied vol is rich versus realized.** Data: Unusual Whales 2-year daily price / implied vol history. Methods: day-by-day path simulation with exits and fees, month-clustered bootstrap, permutation tests, regression, logistic regression with walk-forward scoring, PCA, Monte Carlo account paths, and an in-sample / out-of-sample grid check.

| Section | Question |
|---|---|
| 1 | Does a regression forecast of realized vol (HAR) beat the screener's blend and IV? |
| 2 | Spread simulation: delta x exit rule, after fees and slippage |
| 3 | Does the rich-IV filter help? (bootstrap + permutation test) |
| 4 | Which features explain P&L? (regression, clustered by month) |
| 5 | Is the delta a fair breach probability? (logit, calibration, walk-forward) |
| 6 | PCA: how many independent bets are in the universe? |
| 7 | Monte Carlo: what can a $3,000 account expect over a year? |
| 8 | Is picking the best setting just noise? (in-sample vs out-of-sample) |

**Honest limits.** One flat IV for both strikes (no skew), prices are model values from Black-Scholes, not real quotes, and two years of data with overlapping trades is a small effective sample. Trades opened on the same date share market shocks, so every interval here is **clustered by month**. Use the results to decide *what to log and test live*, not as a promise of returns. Run with `UW_API_KEY` set for live data; without it the notebook uses synthetic data (labelled) just to prove the code runs.

In [ ]:
import os, json, math, time, warnings, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, rankdata, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 50); pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": .25})

# ---------------- settings ----------------
UNIVERSE = ["SPY","QQQ","IWM","DIA","XLF","XLE","XLK","XLV","XLY","XLP","XLU","XLI","XLB","GLD","SLV","TLT","HYG","EEM","EFA","KRE","SMH","XBI"]
DATA_MODE = "auto"              # "auto": live if UW_API_KEY set, else synthetic
CACHE_DIR = Path("data_uw")
ACCOUNT = 3000.0
WIDTH = 1.0                     # $ width of the spread
DTE_MAIN, DTE_ALT = 30, 45      # calendar days to expiry
DELTAS = [0.16, 0.20, 0.25, 0.30]
BASE_DELTA, BASE_EXIT = 0.25, "tp50_dte21"   # the pre-chosen "main" setting (25 delta: the lowest delta that usually clears the $0.20 credit floor); section 8 checks it was not cherry-picked
RF, STEP = 0.04, 5              # risk-free rate; enter every STEP trading days per ticker
SLIP_OPEN, SLIP_CLOSE = 0.02, 0.02   # $ per share (per spread) given up vs mid when opening / closing early
FEE_OPEN, FEE_CLOSE = 2.00, 0.00     # $ per spread ($1 per leg to open, $0 to close; check tastytrade's current schedule)
MIN_CREDIT = 0.20               # net credit floor per $1-wide spread (before fees)
MAX_LOSS_USD = 150.0            # per-trade max loss ceiling (5% of the account)
RICH_AT, THIN_BELOW, IVR_HI = 1.20, 0.95, 50
N_BOOT, N_PERM, SEED = 2000, 1000, 11
ANN = np.sqrt(252)
RULES = ["expiry", "tp50", "dte21", "tp50_dte21", "tp50_stop_dte21"]
RULE_TEXT = {"expiry": "hold to expiry", "tp50": "take 50% profit else expiry", "dte21": "close at 21 DTE",
             "tp50_dte21": "50% profit or 21 DTE", "tp50_stop_dte21": "50% profit, stop at 2x credit, or 21 DTE"}
rng_global = np.random.RandomState(SEED)

## Data
Same source as the first notebook: Unusual Whales `volatility/realized` (2 years of daily close, 30-day implied vol, and the realized vol that followed).

In [ ]:
BASE = "https://api.unusualwhales.com"
def fetch_uw_realized(ticker, key, timeframe="2Y", days=21):
    url = f"{BASE}/api/stock/{ticker}/volatility/realized?timeframe={timeframe}&days={days}"
    req = urllib.request.Request(url, headers={"Authorization": "Bearer " + key, "Accept": "application/json", "User-Agent": "screener-research/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        j = json.loads(r.read().decode("utf-8", "replace"))
    return j.get("data") if isinstance(j, dict) else j

def tidy(rows, ticker):
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None).dt.normalize()
    for c in ("price", "implied_volatility", "realized_volatility"): df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values("date").drop_duplicates("date"); df["ticker"] = ticker
    return df[["date", "ticker", "price", "implied_volatility", "realized_volatility"]].reset_index(drop=True)

def load_live(tickers, key):
    CACHE_DIR.mkdir(exist_ok=True); frames, failed = [], []
    for t in tickers:
        f = CACHE_DIR / f"{t}.csv"
        if f.exists() and time.time() - f.stat().st_mtime < 20 * 3600:
            frames.append(pd.read_csv(f, parse_dates=["date"])); continue
        try:
            df = tidy(fetch_uw_realized(t, key), t); df.to_csv(f, index=False); frames.append(df); time.sleep(0.3)
        except urllib.error.HTTPError as e:
            failed.append((t, e.code))
            if e.code == 401: raise SystemExit("401: Unusual Whales rejected the key. Check UW_API_KEY.")
        except Exception as e:
            failed.append((t, type(e).__name__))
    if failed: print("skipped:", failed)
    return pd.concat(frames, ignore_index=True)

def synth_universe(tickers, seed, n=501, h=21):
    """Synthetic stand-in: shared market vol factor, t-distributed returns, IV about 10% above subsequent vol. Code check only."""
    rng = np.random.RandomState(seed); m = n + 80
    f = np.zeros(m); zm = rng.standard_t(5, m) / np.sqrt(5 / 3)
    for i in range(1, m): f[i] = f[i-1] * 0.97 + 0.07 * rng.randn()
    dates = pd.bdate_range(end="2026-10-23", periods=m); out = []
    for k, t in enumerate(tickers):
        mu = np.log(rng.uniform(0.12, 0.30)); b = rng.uniform(0.4, 1.0); u = np.zeros(m)
        for i in range(1, m): u[i] = u[i-1] * 0.96 + 0.06 * rng.randn()
        vol = np.exp(mu + b * f + u); zi = rng.standard_t(5, m) / np.sqrt(5 / 3)
        r = vol / np.sqrt(252) * (0.6 * zm + 0.8 * zi) + 0.0003
        price = 100 * np.exp(np.cumsum(r))
        iv = (0.6 * vol + 0.4 * np.exp(mu)) * 1.10 * np.exp(0.05 * rng.randn(m))
        fwd = pd.Series(r).rolling(h).std().shift(-h) * np.sqrt(252)
        out.append(pd.DataFrame({"date": dates, "ticker": t, "price": price, "implied_volatility": iv, "realized_volatility": fwd.values}).iloc[40:40 + n])
    return pd.concat(out, ignore_index=True)

key = os.environ.get("UW_API_KEY", "").strip()
mode = DATA_MODE if DATA_MODE != "auto" else ("live" if key else "synthetic")
if mode == "live": raw = load_live(UNIVERSE, key)
else:
    raw = synth_universe(UNIVERSE, SEED)
    print("=" * 78, "\nSYNTHETIC DATA: this only checks that the code runs. Do not draw conclusions.\n" + "=" * 78)
IS_SYNTH = (mode != "live")
print(mode, "| tickers:", raw.ticker.nunique(), "| rows:", len(raw), "|", raw.date.min().date(), "to", raw.date.max().date())

In [ ]:
def build_features(df):
    out = []
    for t, g in df.groupby("ticker", sort=False):
        g = g.sort_values("date").reset_index(drop=True).copy()
        g["ret"] = np.log(g["price"]).diff()
        for w in (5, 21, 63): g[f"rv{w}"] = g["ret"].rolling(w).std() * ANN
        g["rv252"] = g["ret"].rolling(252, min_periods=200).std() * ANN
        g["blend"] = 0.5 * g.rv21 + 0.3 * g.rv63 + 0.2 * g.rv252          # the screener's blend
        g["blend_s"] = 0.6 * g.rv21 + 0.4 * g.rv63                           # shorter blend, available after ~3 months
        g["ewma"] = np.sqrt((g["ret"] ** 2).ewm(alpha=0.06, adjust=False).mean() * 252); g.loc[:20, "ewma"] = np.nan
        g["iv"] = g["implied_volatility"]; g["fwd_rv"] = g["realized_volatility"]
        lo, hi = g.iv.rolling(252, min_periods=126).min(), g.iv.rolling(252, min_periods=126).max()
        g["ivr"] = 100 * (g.iv - lo) / (hi - lo)
        g["ret5"] = g["price"].pct_change(5)
        out.append(g)
    F = pd.concat(out, ignore_index=True)
    mk = F[F.ticker == "SPY"].set_index("date").iv if "SPY" in set(F.ticker) else F.groupby("date").iv.median()
    F["iv_mkt"] = F.date.map(mk)
    return F
F = build_features(raw)
print("feature rows:", len(F), "| rows with finished forward RV:", int(F.fwd_rv.notna().sum()))

## 1. Does a regression forecast of realized vol (HAR) beat the screener's blend?
HAR-RV is the standard simple regression forecast: next month's realized vol regressed on short, medium and long trailing realized vol (here 5, 21, 63, 252 days), pooled across tickers and **refit monthly using only data whose outcome was already known**. It is the regression version of the screener's 0.5/0.3/0.2 blend, with weights learned instead of guessed. Scored by QLIKE (lower is better) and Mincer-Zarnowitz R2 on identical rows.

In [ ]:
def har_walk_forward(F, cols=("rv5", "rv21", "rv63", "rv252"), refit_every=21, min_train=400):
    F = F.copy(); F["har"] = np.nan; cols = list(cols)
    d = F.dropna(subset=cols).copy(); dates = np.sort(d.date.unique()); refits = dates[::refit_every]; coefs = []
    for k, rd in enumerate(refits):
        rd = pd.Timestamp(rd)
        tr = d[(d.date <= rd - pd.Timedelta(days=32)) & d.fwd_rv.notna()]     # outcome fully known before the refit date
        if len(tr) < min_train: continue
        res = sm.OLS(tr.fwd_rv, sm.add_constant(tr[cols])).fit(); coefs.append(dict(date=rd, **res.params.to_dict()))
        nxt = pd.Timestamp(refits[k + 1]) if k + 1 < len(refits) else pd.Timestamp(dates[-1]) + pd.Timedelta(days=1)
        m = (d.date >= rd) & (d.date < nxt)
        F.loc[d.index[m], "har"] = np.clip(res.predict(sm.add_constant(d.loc[m, cols], has_constant="add")), 0.03, None)
    return F, pd.DataFrame(coefs)

F, har_coefs = har_walk_forward(F)
F["ratio_rv21"] = F.iv / F.rv21; F["ratio_blend"] = F.iv / F.blend; F["ratio_s"] = F.iv / F.blend_s; F["ratio_har"] = F.iv / F.har
F["rel_ratio"] = F.ratio_s - F.groupby("date").ratio_s.transform("median")     # richness relative to the other tickers on the same day
if len(har_coefs): print("average HAR coefficients:", har_coefs.drop(columns="date").mean().round(3).to_dict())

cands = ["iv", "rv21", "blend", "blend_s", "ewma", "har"]
ev = F.dropna(subset=["fwd_rv"] + cands).copy()
qlike = lambda y, f: (lambda r: r - np.log(r) - 1)((y / f) ** 2)
rows = []
for c in cands:
    mz = sm.OLS(ev.fwd_rv, sm.add_constant(ev[c])).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(ev.ticker)[0]})
    rows.append(dict(forecast=c, MAE_pts=100 * (ev[c] - ev.fwd_rv).abs().mean(), bias_pts=100 * (ev[c] - ev.fwd_rv).mean(), QLIKE=qlike(ev.fwd_rv, ev[c]).mean(), MZ_R2=mz.rsquared))
res1 = pd.DataFrame(rows).set_index("forecast").sort_values("QLIKE")
print("scored rows:", len(ev), "|", ev.date.min().date(), "to", ev.date.max().date()); display(res1)
HAR_BEATS_BLEND = bool(res1.QLIKE.get("har", 9) < res1.QLIKE.get("blend", 0))

## 2. Spread simulation: delta x exit rule, after fees and slippage
For every ticker, every `STEP` trading days: sell a put at the target delta (strike rounded to a listed strike: $0.50 steps under $60, $1 otherwise), buy the put `WIDTH` below, then **re-price the spread every day** with Black-Scholes at that day's IV and the remaining time. That lets the exit rules (50% profit, 21 days to expiry, stop at 2x credit) be tested on the same path. Costs: `SLIP_OPEN`/`SLIP_CLOSE` per share relative to mid, `FEE_OPEN` to open. A trade counts as **eligible** if it meets the credit and max-loss rules from your rule set.

In [ ]:
def bs_put(S, K, T, sig, r=RF):
    S, T, sig = np.asarray(S, float), np.asarray(T, float), np.asarray(sig, float)
    with np.errstate(all="ignore"):
        d1 = (np.log(S / K) + (r + sig ** 2 / 2) * T) / (sig * np.sqrt(T)); d2 = d1 - sig * np.sqrt(T)
        v = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return np.where(T > 1e-9, v, np.maximum(K - S, 0))

def short_strike(S, T, sig, delta, r=RF):
    return S * np.exp((r + sig ** 2 / 2) * T - norm.ppf(1 - delta) * sig * np.sqrt(T))

def strike_step(S): return 0.5 if S < 60 else 1.0

def exit_outcomes(V, credit, J):
    H = len(V) - 1; c = lambda j: min(V[j] + SLIP_CLOSE, WIDTH)
    first = lambda cond, lo, hi: next((j for j in range(lo, hi) if cond(j)), None)
    out = {"expiry": (V[H], False)}
    tp = first(lambda j: c(j) <= 0.5 * credit, 1, H); out["tp50"] = (c(tp), True) if tp is not None else (V[H], False)
    out["dte21"] = (c(J), True)
    tp2 = first(lambda j: c(j) <= 0.5 * credit, 1, J); out["tp50_dte21"] = (c(tp2), True) if tp2 is not None else (c(J), True)
    ev_ = first(lambda j: c(j) <= 0.5 * credit or c(j) >= 3 * credit, 1, J); out["tp50_stop_dte21"] = (c(ev_), True) if ev_ is not None else (c(J), True)
    return out

FEATS = ["ratio_rv21", "ratio_blend", "ratio_s", "ratio_har", "rel_ratio", "ivr", "ret5", "iv_mkt"]
def simulate(F, delta, dte):
    H = int(round(dte * 252 / 365)); T0 = dte / 365; J = int(round(H * (dte - 21) / dte)); rows = []
    for t, g in F.groupby("ticker", sort=False):
        g = g.sort_values("date").reset_index(drop=True); P = g.price.to_numpy(); IV = g.iv.to_numpy()
        for i in range(0, len(g) - H, STEP):
            S0, s0 = P[i], IV[i]
            if not (np.isfinite(S0) and np.isfinite(s0) and s0 > 0.03): continue
            step = strike_step(S0); Ks = round(float(short_strike(S0, T0, s0, delta)) / step) * step; Kl = Ks - WIDTH
            if Kl <= 0: continue
            ivp = pd.Series(IV[i:i + H + 1]).ffill().to_numpy()
            if not np.all(np.isfinite(ivp)): continue
            j = np.arange(H + 1); Tj = T0 * (H - j) / H
            V = np.clip(bs_put(P[i:i + H + 1], Ks, Tj, ivp) - bs_put(P[i:i + H + 1], Kl, Tj, ivp), 0, WIDTH)
            credit = V[0] - SLIP_OPEN
            if credit <= 0.02 or credit >= WIDTH: continue
            d1 = (np.log(S0 / Ks) + (RF + s0 ** 2 / 2) * T0) / (s0 * math.sqrt(T0)); d2 = d1 - s0 * math.sqrt(T0)
            rec = dict(ticker=t, date=g.date[i], delta_target=delta, dte=dte, S=S0, iv=s0, Ks=Ks, credit=credit, delta_actual=norm.cdf(-d1),
                       nominal_breach=norm.cdf(-d2), breached=bool(P[i + H] < Ks), maxloss=(WIDTH - credit) * 100 + FEE_OPEN)
            for f_ in FEATS: rec[f_] = g.at[i, f_]
            for rule, (cost, early) in exit_outcomes(V, credit, J).items():
                rec["pnl_" + rule] = (credit - cost) * 100 - FEE_OPEN - (FEE_CLOSE if early else 0); rec["early_" + rule] = early
            rows.append(rec)
    return pd.DataFrame(rows)

t0 = time.time()
TR = pd.concat([simulate(F, d, DTE_MAIN) for d in DELTAS] + [simulate(F, BASE_DELTA, DTE_ALT)], ignore_index=True)
TR["month"] = TR.date.dt.to_period("M").astype(str)
TR["eligible"] = (TR.credit >= MIN_CREDIT) & (TR.maxloss <= MAX_LOSS_USD)
print("simulated %d trades in %.0fs | eligible (credit >= $%.2f, max loss <= $%.0f): %d (%.0f%%)" % (len(TR), time.time() - t0, MIN_CREDIT, MAX_LOSS_USD, TR.eligible.sum(), 100 * TR.eligible.mean()))
print("eligible trades per delta target (low deltas often fail the credit floor on a $1-wide spread, which is itself a finding):"); print(TR[TR.dte == DTE_MAIN].groupby("delta_target").eligible.agg(["size", "sum", "mean"]).rename(columns={"size": "all", "sum": "eligible", "mean": "share"}))

In [ ]:
def cboot(df, col, a=None, b=None, n=N_BOOT, seed=SEED):
    """Month-clustered bootstrap of a mean (or of a difference of means, a minus b). Resamples whole months so trades opened on the same
    dates, which share market shocks, stay together."""
    months = np.sort(df.month.unique()); pos = {m: i for i, m in enumerate(months)}
    def agg(mask):
        d = df if mask is None else df[mask]; s = np.zeros(len(months)); c = np.zeros(len(months))
        for m, (ss, cc) in d.groupby("month")[col].agg(["sum", "count"]).iterrows(): s[pos[m]] = ss; c[pos[m]] = cc
        return s, c
    sa, ca = agg(a); idx = np.random.RandomState(seed).randint(0, len(months), (n, len(months)))
    ma = sa[idx].sum(1) / np.maximum(ca[idx].sum(1), 1); point = sa.sum() / max(ca.sum(), 1)
    if b is not None:
        sb, cb = agg(b); mb = sb[idx].sum(1) / np.maximum(cb[idx].sum(1), 1); ma = ma - mb; point -= sb.sum() / max(cb.sum(), 1)
    lo, hi = np.percentile(ma, [2.5, 97.5]); return point, lo, hi

def trade_stats(d, col, label):
    p, lo, hi = cboot(d, col); w = d[col] > 0
    return dict(setting=label, n=len(d), mean_pnl=p, ci_lo=lo, ci_hi=hi, win_pct=100 * w.mean(), avg_win=d[col][w].mean(), avg_loss=d[col][~w].mean(), worst=d[col].min())

E = TR[(TR.dte == DTE_MAIN) & TR.eligible]
rows = []
for dl in DELTAS:
    for rule in RULES:
        if (E.delta_target == dl).sum() < 20: continue      # too few trades clear the credit floor at this delta to say anything
        rows.append(trade_stats(E[E.delta_target == dl], "pnl_" + rule, f"delta {dl:.2f} | {RULE_TEXT[rule]}") | dict(delta=dl, rule=rule))
G = pd.DataFrame(rows).set_index("setting")
display(G.drop(columns=["delta", "rule"]))
b = E[E.delta_target == BASE_DELTA]
print(f"\nBASE setting (delta {BASE_DELTA}, {RULE_TEXT[BASE_EXIT]}): actual breach {100 * b.breached.mean():.1f}% vs delta-implied {100 * b.nominal_breach.mean():.1f}%  |  trades closed early {100 * b['early_' + BASE_EXIT].mean():.0f}%")
p, lo, hi = cboot(b, "pnl_" + BASE_EXIT); BASE_MEAN = (p, lo, hi)
alt = TR[(TR.dte == DTE_ALT) & TR.eligible]
if len(alt): print(f"Same rule at {DTE_ALT} DTE: mean ${cboot(alt, 'pnl_' + BASE_EXIT)[0]:.2f} per spread over {len(alt)} trades (vs ${p:.2f} at {DTE_MAIN} DTE)")
fig, ax = plt.subplots(); best = G.reset_index().pivot(index="rule", columns="delta", values="mean_pnl").loc[RULES]
best.plot.bar(ax=ax); ax.axhline(0, color="k", lw=.8); ax.set_ylabel("mean P&L per spread ($)"); ax.set_xlabel(""); ax.set_title("Mean P&L per $1-wide spread by delta and exit rule (after costs)"); plt.xticks(rotation=20); plt.show()

## 3. Does the rich-IV filter help?
For the base setting, compare trades taken when a signal says "rich" with trades where it does not. Cut-offs are fixed in advance (1.20 for the IV/HV ratios, +0.15 above the same-day median for relative richness, IV rank 50). Two checks: a **month-clustered bootstrap interval** on the difference in mean P&L, and a **permutation test** that shuffles the signal labels in 4-trade blocks within each ticker and asks how often chance alone produces a difference this large.

In [ ]:
SIGS = {"IV / realized 21d >= 1.20": ("ratio_rv21", lambda s: s >= RICH_AT), "IV / screener blend >= 1.20": ("ratio_blend", lambda s: s >= RICH_AT),
        "IV / short blend >= 1.20": ("ratio_s", lambda s: s >= RICH_AT), "IV / HAR forecast >= 1.20": ("ratio_har", lambda s: s >= RICH_AT),
        "rich vs other tickers (+0.15)": ("rel_ratio", lambda s: s >= 0.15), "IV rank >= 50": ("ivr", lambda s: s >= IVR_HI)}
col = "pnl_" + BASE_EXIT

def perm_test(d, flag, n=N_PERM, block=4, seed=SEED):
    rng = np.random.RandomState(seed); parts = []
    for _, g in d.sort_values("date").groupby("ticker"):
        parts.append((g[col].to_numpy(), flag[g.index].to_numpy() if hasattr(flag, "index") else None))
    pn = np.concatenate([p for p, _ in parts]); fl = np.concatenate([f for _, f in parts]); obs = pn[fl].mean() - pn[~fl].mean()
    sizes = [len(p) for p, _ in parts]; stats = np.empty(n)
    for k in range(n):
        shuffled = []
        for (p, f) in parts:
            nb = len(f) // block; head = f[:nb * block].reshape(nb, block)[rng.permutation(nb)].ravel() if nb > 1 else f[:nb * block]
            shuffled.append(np.concatenate([head, f[nb * block:]]))
        s = np.concatenate(shuffled); stats[k] = pn[s].mean() - pn[~s].mean() if s.any() and (~s).any() else 0
    return obs, float((np.abs(stats) >= abs(obs)).mean())

rows = []
for name, (c, fn) in SIGS.items():
    d = b.dropna(subset=[c]).copy().reset_index(drop=True)
    if len(d) < 40: continue
    flag = pd.Series(fn(d[c]).to_numpy(), index=d.index); 
    if flag.sum() < 15 or (~flag).sum() < 15: continue
    diff, lo, hi = cboot(d, col, a=flag.values, b=~flag.values)
    obs, pval = perm_test(d, flag)
    rows.append(dict(signal=name, n_signal=int(flag.sum()), n_other=int((~flag).sum()), mean_when_signal=d[col][flag].mean(), mean_otherwise=d[col][~flag].mean(),
                     diff=diff, ci_lo=lo, ci_hi=hi, perm_p=pval))
sig_tab = pd.DataFrame(rows).set_index("signal"); display(sig_tab)
SIG_SURVIVORS = list(sig_tab.index[(sig_tab.ci_lo > 0) & (sig_tab.perm_p < 0.05)]) if len(sig_tab) else []
print("signals whose interval excludes 0 AND permutation p < 0.05:", SIG_SURVIVORS or "none")
print("(With several signals tested, expect about one false positive at p<0.05 by luck alone; treat a single borderline result as a lead, not a finding.)")

## 4. Which features explain P&L? (regression)
OLS on the base setting's return on risk (P&L / max loss) with standardized features and **ticker fixed effects**, standard errors clustered by month. Features: the short-blend IV/HV ratio, richness relative to other tickers (a one-factor, PCA-like demeaning), IV rank, the last 5 days' return, and the market IV level (SPY). Coefficients are in return-on-risk per one standard deviation.

In [ ]:
d4 = b.dropna(subset=["ratio_s", "rel_ratio", "ivr", "ret5", "iv_mkt"]).copy()
d4["ror"] = d4[col] / d4.maxloss
for c in ["ratio_s", "rel_ratio", "ivr", "ret5", "iv_mkt"]: d4[c + "_z"] = (d4[c] - d4[c].mean()) / d4[c].std()
m4 = smf.ols("ror ~ ratio_s_z + rel_ratio_z + ivr_z + ret5_z + iv_mkt_z + C(ticker)", data=d4).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d4.month)[0]})
tab4 = pd.DataFrame({"coef": m4.params, "se": m4.bse, "t": m4.tvalues, "ci_lo": m4.conf_int()[0], "ci_hi": m4.conf_int()[1]}).loc[[i for i in m4.params.index if i.endswith("_z")]]
display(tab4); print(f"n={int(m4.nobs)} trades | clusters (months)={d4.month.nunique()} | R2={m4.rsquared:.3f} (low R2 is normal: most of a spread's outcome is luck)")
REG_SIG = list(tab4.index[tab4.t.abs() > 2])
print("features with |t| > 2:", REG_SIG or "none")

## 5. Is the delta a fair breach probability? (logistic regression)
If a 20-delta put were a fair 20% chance of finishing in the money, a logit of "was breached" on the model's own breach probability would have slope near 1 and intercept near 0. Then test whether adding features (IV/HV, IV rank, recent return) improves forecasts **out of sample**: fit on the earlier 60% of dates, score the later 40% (with a 30-day gap), compare Brier scores (lower is better) and AUC against the plain delta.

In [ ]:
d5 = TR[(TR.dte == DTE_MAIN) & TR.eligible].dropna(subset=["ratio_s", "ivr", "ret5"]).copy()
d5["y"] = d5.breached.astype(int); d5["logit_nom"] = np.log(d5.nominal_breach / (1 - d5.nominal_breach))
for c in ["ratio_s", "ivr", "ret5"]: d5[c + "_z"] = (d5[c] - d5[c].mean()) / d5[c].std()
try:
    m5 = smf.logit("y ~ logit_nom", data=d5).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(d5.month)[0]})
    ci = m5.conf_int().loc["logit_nom"]
    print(f"breach ~ logit(model probability): slope {m5.params['logit_nom']:.2f} (95% CI {ci[0]:.2f} to {ci[1]:.2f}), intercept {m5.params['Intercept']:.2f}")
    LOGIT_SLOPE = m5.params["logit_nom"]
except Exception as e:
    print("logit failed:", e); LOGIT_SLOPE = np.nan
d5["bin"] = pd.qcut(d5.nominal_breach, 5, duplicates="drop")
cal = d5.groupby("bin").agg(trades=("y", "size"), model_pct=("nominal_breach", lambda s: 100 * s.mean()), actual_pct=("y", lambda s: 100 * s.mean())); display(cal)

split = d5.date.quantile(0.6); tr_, te_ = d5[d5.date < split - pd.Timedelta(days=30)], d5[d5.date >= split]
def auc(y, p):
    r = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan
try:
    mm = smf.logit("y ~ logit_nom + ratio_s_z + ivr_z + ret5_z", data=tr_).fit(disp=0)
    pm = mm.predict(te_); pn_ = te_.nominal_breach; base_rate = tr_.y.mean()
    wf = pd.DataFrame({"Brier": [((pn_ - te_.y) ** 2).mean(), ((pm - te_.y) ** 2).mean(), ((base_rate - te_.y) ** 2).mean()], "AUC": [auc(te_.y.values, pn_.values), auc(te_.y.values, pm.values), 0.5]},
                      index=["plain delta", "delta + features", "always the average breach rate"])
    print(f"walk-forward: fit on {len(tr_)} trades before {split.date()}, scored on {len(te_)} later trades"); display(wf)
    print("features coefficients (training fit):", mm.params.round(2).to_dict())
    WF_HELPS = bool(wf.Brier.iloc[1] < wf.Brier.iloc[0] - 1e-4)
except Exception as e:
    print("walk-forward logit failed:", e); WF_HELPS = False

## 6. PCA: how many independent bets are in the universe?
Ten ETF spreads are not ten bets if the ETFs move together. PCA on the daily changes in log implied vol shows how much of IV movement is one market-wide factor; the **participation ratio** of the return correlation matrix, (sum of eigenvalues)^2 / sum of squared eigenvalues, is the effective number of independent return streams.

In [ ]:
piv_iv = F.pivot(index="date", columns="ticker", values="iv"); dl = np.log(piv_iv).diff().dropna(how="all")
dl = dl.dropna(axis=1, thresh=int(0.9 * len(dl))).fillna(0)
w, v = np.linalg.eigh(np.corrcoef(dl.values.T)); o = np.argsort(w)[::-1]; w, v = w[o], v[:, o]
if v[:, 0].sum() < 0: v[:, 0] *= -1
share = w / w.sum()
print("IV changes: PC1 explains %.0f%% of movement, PC2 %.0f%%, PC3 %.0f%% (a single market-wide vol factor if PC1 is large)" % tuple(100 * share[:3]))
loads = pd.DataFrame({"PC1_loading": v[:, 0], "share_of_own_IV_moves_from_PC1": v[:, 0] ** 2 * w[0]}, index=dl.columns).sort_values("share_of_own_IV_moves_from_PC1"); display(loads)
piv_r = F.pivot(index="date", columns="ticker", values="ret").dropna(how="any", axis=0); wr = np.linalg.eigvalsh(np.corrcoef(piv_r.values.T))
N_EFF = float(wr.sum() ** 2 / (wr ** 2).sum()); print(f"returns: {piv_r.shape[1]} tickers behave like about {N_EFF:.1f} independent bets (participation ratio)")
fig, ax = plt.subplots(); ax.bar(range(1, len(share) + 1), 100 * share); ax.set_xlabel("principal component of daily IV changes"); ax.set_ylabel("% of variance"); ax.set_title("Section 6: scree plot"); plt.show()

## 7. Monte Carlo: a year on a $3,000 account
Resamples **chronological blocks of 4 eligible trades** from the base setting, so streaks and shared-date shocks survive, and plays out about 40 one-lot trades (roughly one every 1 to 2 weeks on one or two names). Three cases: as simulated, with **extra slippage** of $0.03 a spread, and the same with the filter applied (only trades flagged rich by the short blend). Also reports the **break-even slippage**: how many cents per spread of extra cost it takes to wipe out the mean P&L.

In [ ]:
def run_mc(pnl, n_trades=40, block=4, paths=5000, seed=SEED):
    rng = np.random.RandomState(seed); pnl = np.asarray(pnl, float); K = int(np.ceil(n_trades / block))
    st = rng.randint(0, len(pnl) - block + 1, (paths, K)); idx = (st[:, :, None] + np.arange(block)).reshape(paths, -1)[:, :n_trades]
    eq = ACCOUNT + np.cumsum(pnl[idx], axis=1); peak = np.maximum.accumulate(np.hstack([np.full((paths, 1), ACCOUNT), eq]), axis=1)[:, 1:]
    return eq, (peak - eq).max(axis=1)

bs = b.sort_values(["date", "ticker"]); early = bs["early_" + BASE_EXIT].astype(float).to_numpy()
cases = {"as simulated": bs[col].to_numpy(),
         "extra slippage $0.03": bs[col].to_numpy() - 100 * 0.03 * (1 + early)}
rich = bs[bs.ratio_s >= RICH_AT]
if len(rich) > 30: cases["rich filter only (IV/short blend >= 1.20)"] = rich[col].to_numpy()
rows, fans = [], {}
for name, pnl in cases.items():
    eq, dd = run_mc(pnl); end = eq[:, -1]; fans[name] = eq
    rows.append(dict(case=name, trades_in_pool=len(pnl), mean_per_trade=pnl.mean(), median_end=np.median(end), p5_end=np.percentile(end, 5), p95_end=np.percentile(end, 95),
                     prob_loss_pct=100 * (end < ACCOUNT).mean(), prob_drawdown_over_10pct=100 * (dd > 0.10 * ACCOUNT).mean(), prob_drawdown_over_20pct=100 * (dd > 0.20 * ACCOUNT).mean()))
mc = pd.DataFrame(rows).set_index("case"); display(mc)
mean0, early_rate = bs[col].mean(), early.mean()
BE_SLIP = mean0 / (100 * (1 + early_rate))
if BE_SLIP > 0: print(f"break-even extra slippage: ${BE_SLIP:.3f} per share per spread (about {BE_SLIP * 100:.1f} cents) before the mean P&L reaches zero; the model already assumes ${SLIP_OPEN:.2f} to open and ${SLIP_CLOSE:.2f} to close")
else: print("mean P&L is already at or below zero after the modeled costs, so there is no slippage cushion")
fig, ax = plt.subplots(); eq = fans["as simulated"]; x = np.arange(1, eq.shape[1] + 1)
ax.fill_between(x, np.percentile(eq, 5, axis=0), np.percentile(eq, 95, axis=0), alpha=.2); ax.plot(x, np.median(eq, axis=0), label="median"); ax.axhline(ACCOUNT, color="k", lw=.8)
ax.set_xlabel("trade number"); ax.set_ylabel("account value ($)"); ax.set_title("Section 7: one year of 1-lot trades (5th to 95th percentile)"); ax.legend(); plt.show()

## 8. Is picking the best setting just noise?
Pick a "best" delta and exit rule from the first half of the data, then see how it does on the second half (with a 30-day gap so no trade straddles the split). If settings that look best early are not best later, the grid in section 2 is mostly noise and the right move is to pick a sensible setting and stick with it. Also shows the result by half-year to expose regime dependence.

In [ ]:
sp = E.date.quantile(0.5); IS_, OOS_ = E[E.date < sp - pd.Timedelta(days=30)], E[E.date >= sp]
rows = [dict(delta=dl, rule=rl, IS=IS_[IS_.delta_target == dl]["pnl_" + rl].mean(), OOS=OOS_[OOS_.delta_target == dl]["pnl_" + rl].mean()) for dl in DELTAS for rl in RULES]
wf8 = pd.DataFrame(rows).dropna()
rho = spearmanr(wf8.IS, wf8.OOS).correlation; pick = wf8.loc[wf8.IS.idxmax()]
display(wf8.sort_values("IS", ascending=False).head(8))
print(f"rank correlation of in-sample vs out-of-sample mean P&L across {len(wf8)} settings: {rho:.2f}")
print(f"best in-sample: delta {pick.delta}, '{RULE_TEXT[pick.rule]}': in-sample ${pick.IS:.2f} -> out-of-sample ${pick.OOS:.2f}   |   average OOS across all settings: ${wf8.OOS.mean():.2f}")
fig, ax = plt.subplots(); ax.scatter(wf8.IS, wf8.OOS); ax.axhline(0, color="k", lw=.7); ax.axvline(0, color="k", lw=.7); ax.set_xlabel("mean P&L first half ($)"); ax.set_ylabel("mean P&L second half ($)"); ax.set_title("Section 8: does early performance persist?"); plt.show()
E2 = b.copy(); E2["half"] = E2.date.dt.year.astype(str) + "H" + np.where(E2.date.dt.month <= 6, "1", "2")
display(E2.groupby("half").agg(trades=(col, "size"), mean_pnl=(col, "mean"), win_pct=(col, lambda s: 100 * (s > 0).mean()), breach_pct=("breached", lambda s: 100 * s.mean())))

## Summary

In [ ]:
print("DATA:", "SYNTHETIC (code check only)" if IS_SYNTH else "Unusual Whales live history", f"| {F.ticker.nunique()} tickers | {len(TR)} simulated trades\n")
p, lo, hi = BASE_MEAN
print(f"Base setting: delta {BASE_DELTA}, {RULE_TEXT[BASE_EXIT]}, {DTE_MAIN} DTE, ${WIDTH:.0f} wide")
print(f"  mean P&L per spread: ${p:.2f} (95% month-clustered CI ${lo:.2f} to ${hi:.2f}) after fees and slippage; break-even extra slippage {max(BE_SLIP, 0) * 100:.1f} cents")
print("  verdict:", "positive after costs (interval above zero)" if lo > 0 else "NEGATIVE after costs (interval below zero)" if hi < 0 else "not distinguishable from zero (interval includes zero)")
print(f"Regression forecast beats the screener blend: {'yes' if HAR_BEATS_BLEND else 'no'}  (see section 1)")
print(f"Rich-IV signals that pass both tests: {SIG_SURVIVORS or 'none'}")
print(f"Regression features with |t| > 2: {REG_SIG or 'none'}")
print(f"Delta calibration slope: {LOGIT_SLOPE:.2f} (1.0 means the delta is a fair breach probability); features improve out-of-sample breach forecasts: {'yes' if WF_HELPS else 'no'}")
print(f"Independent bets in the universe: about {N_EFF:.1f} of {piv_r.shape[1]}")
r0 = mc.iloc[0]; print(f"Year of 40 one-lots: median ending ${r0.median_end:,.0f}, 5th percentile ${r0.p5_end:,.0f}, chance of finishing below ${ACCOUNT:,.0f}: {r0.prob_loss_pct:.0f}%")
print(f"Does the best early setting stay best? rank correlation {rho:.2f}; best-in-sample setting out of sample: ${pick.OOS:.2f}")
print("""
How to use this:
- If the base interval includes zero, the honest conclusion is 'not proven'. The live account's job is then to measure real fills and decide.
- Trust a signal only if it passes both the bootstrap and the permutation test and still matters in the regression.
- If in-sample and out-of-sample rankings do not agree, do not tune the settings. Pick one and log trades.
- Costs matter most at this size: watch the break-even slippage against what tastytrade actually gives you at the mid.
- Next: replace the flat-IV model with real chain snapshots (log them daily) and rerun on 50 or more tickers.
""")